# Fase 1 — Preparação dos dados


In [1]:
import pandas as pd

from scouting.carregar import carregar_fbref

pd.set_option("display.max_columns", 50)

In [2]:
fbref = carregar_fbref()
print(f"{len(fbref)} linhas, {fbref.shape[1]} colunas")
fbref.head()

2854 linhas, 40 colunas


,jogador,nacionalidade,posicao,clube,liga,ano_nascimento,jogos,jogos_titular,minutos,gols,gols_penalti,penaltis_cobrados,xg,npxg,finalizacoes,finalizacoes_no_alvo,assistencias,xag,xa,passes_chave,passes_na_area,acoes_que_geram_chute,acoes_que_geram_gol,passes_progressivos,conducoes_progressivas,recepcoes_progressivas,dribles_certos,dribles_tentados,toques,erros_de_dominio,desarmes_sofridos,recuperacoes,desarmes_e_interceptacoes,gols_time_em_campo,gols_sofridos_em_campo,saldo_gols_por90_em_campo,saldo_gols_on_off,xg_time_em_campo,xg_sofrido_em_campo,saldo_xg_por90_em_campo
0,Max Aarons,eng ENG,DF,Bournemouth,Premier League,2000.0,3,1,86,0,0,0,0.0,0.0,0,0,0,0.0,0.0,0,0,2,0,8,1,3,0,2,73,1,0,7,3,2,0,2.09,1.82,2.3,0.3,2.12
1,Max Aarons,eng ENG,"DF,MF",Valencia,La Liga,2000.0,4,1,120,0,0,0,0.0,0.0,0,0,0,0.0,0.0,1,0,1,0,6,0,10,1,4,85,3,1,7,4,1,3,-1.50,-1.28,1.5,3.7,-1.69
2,Rodrigo Abajas,es ESP,DF,Valencia,La Liga,2003.0,1,1,65,0,0,0,0.1,0.1,1,0,0,0.0,0.0,0,0,0,0,2,3,3,1,1,36,0,2,2,4,1,2,-1.38,-1.14,1.4,0.7,0.93
3,James Abankwah,ie IRL,"DF,MF",Udinese,Serie A,2004.0,6,0,88,0,0,0,0.1,0.1,1,0,0,0.0,0.0,0,0,1,0,4,3,1,0,0,65,1,3,7,5,2,0,2.05,2.50,0.6,1.5,-0.91
4,Keyliane Abdallah,fr FRA,FW,Marseille,Ligue 1,2006.0,1,0,3,0,0,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,1,0,0,0,4,1,0,0,1,0,0,0.00,-0.79,0.0,0.0,-0.49


### Quantos jogadores por liga?

In [3]:
fbref["liga"].value_counts()

liga
Serie A           634
La Liga           601
Premier League    574
Ligue 1           553
Bundesliga        492
Name: count, dtype: int64

### Atenção: jogadores repetidos



In [4]:
repetidos = fbref[fbref.duplicated(["jogador", "ano_nascimento"], keep=False)]
print(f"{repetidos['jogador'].nunique()} jogadores aparecem em mais de uma linha")
repetidos.sort_values("jogador")[["jogador", "clube", "liga", "minutos", "gols", "xg"]].head(6)

147 jogadores aparecem em mais de uma linha


,jogador,clube,liga,minutos,gols,xg
1351,Abdukodir Khusanov,Manchester City,Premier League,503,0,0.0
1352,Abdukodir Khusanov,Lens,Ligue 1,975,0,0.2
188,Adam Aznou,Valladolid,La Liga,876,0,0.3
189,Adam Aznou,Bayern Munich,Bundesliga,17,0,0.0
22,Akor Adams,Sevilla,La Liga,134,0,0.4
23,Akor Adams,Montpellier,Ligue 1,1075,3,4.3


**Passo 1.2 — Juntar quem trocou de clube.** Cada jogador passa a ter uma linha só:

- **Contagens** (minutos, gols, xG...): somadas.
- **Saldo por 90 em campo**: recalculado a partir das somas, `(marcados − sofridos) ÷ (minutos ÷ 90)`.
- **On-off**: média ponderada pelos minutos em cada clube (aproximação: a conta exata precisaria dos dados do time sem ele).
- **Clube, liga e posição**: os do clube onde ele jogou mais minutos.

In [5]:
from scouting.limpar import juntar_trocas_de_clube

jogadores = juntar_trocas_de_clube(fbref)
print(f"Antes: {len(fbref)} linhas | Depois: {len(jogadores)} linhas")

Antes: 2854 linhas | Depois: 2707 linhas


### Conferências

Nada pode se perder na junção: o total de minutos e de gols tem que ser igual antes e depois, e nenhum jogador pode aparecer duas vezes.

In [ ]:
assert jogadores["minutos"].sum() == fbref["minutos"].sum()
assert jogadores["gols"].sum() == fbref["gols"].sum()
assert not jogadores.duplicated(["jogador", "ano_nascimento"]).any()
assert len(jogadores) == len(fbref) - (fbref.duplicated(["jogador", "ano_nascimento"]).sum())
print("sem duplicidade de dados.")
print("Jogadores em mais de um clube:", (jogadores["num_clubes"] > 1).sum())

Minutos e gols batem, nenhum jogador duplicado, nenhum perdido.
Jogadores em mais de um clube: 147


### Exemplo: Akor Adams (Sevilla 134 min + Montpellier 1.075 min)

Esperado: 1.209 minutos, saldo por 90 = (13 − 32) ÷ (1.209 ÷ 90) ≈ −1,41, on-off ≈ +0,20.

In [7]:
colunas = ["jogador", "clube_principal", "clubes", "minutos", "gols", "xg",
           "saldo_gols_por90_em_campo", "saldo_gols_on_off"]
jogadores.loc[jogadores["jogador"] == "Akor Adams", colunas].round(2)

,jogador,clube_principal,clubes,minutos,gols,xg,saldo_gols_por90_em_campo,saldo_gols_on_off
1396,Akor Adams,Montpellier,Montpellier / Sevilla,1209,3,4.7,-1.41,0.2
